# Borusyak–Jaravel–Spiess Imputation DiD

**Econometrics Notebook Library · v0.1.0**

## Intuition

Use only untreated observations to learn the untreated outcome process. Predict each treated observation's counterfactual untreated outcome, then average the treated residuals.

In the baseline FE model,

$$Y_{it}(0)=\alpha_i+\lambda_t+X_{it}'\beta+\varepsilon_{it}.$$

Estimate this equation on $D_{it}=0$ only, impute $\widehat Y_{it}(0)$ where $D_{it}=1$, and compute

$$\widehat\tau_{it}=Y_{it}-\widehat Y_{it}(0).$$

The estimand is then an explicit weighted average of these imputed treatment effects.

Reference: [Borusyak, Jaravel & Spiess, Review of Economic Studies (2024)](https://doi.org/10.1093/restud/rdae007).

## Why it is different from naive TWFE

Conventional event-study OLS asks treated observations to help estimate coefficients that implicitly define their own counterfactual comparisons. The imputation estimator first isolates the untreated outcome model using untreated observations. Treatment effects are then formed *after* the counterfactual is estimated.

This cleanly separates assumptions about untreated outcomes from restrictions on treatment-effect heterogeneity.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (8, 4.5)
pd.set_option("display.max_columns", 30)

In [ ]:
from econnotes.core import simulate_staggered_panel, bjs_imputation

df = simulate_staggered_panel(n_units=260, seed=51)
bjs, augmented = bjs_imputation(df)
truth = (df[df.treated.eq(1)]
         .groupby("event_time", as_index=False)["tau_true"].mean()
         .rename(columns={"tau_true":"truth"}))
plot = bjs.merge(truth, on="event_time", how="left")
plot.head(8)

In [ ]:
fig, ax = plt.subplots()
ax.plot(plot.event_time, plot.truth, marker="o", label="Truth")
ax.errorbar(plot.event_time, plot.estimate, yerr=1.96*plot.se, marker="o", capsize=3, label="BJS imputation")
ax.set(xlabel="Event time", ylabel="Effect", title="Counterfactual first, treatment effects second")
ax.legend();

## Diagnostic: untreated residuals

If the untreated outcome model is the identifying engine, its residual structure deserves direct scrutiny. Large systematic residual patterns by time or cohort should not be hidden behind the treatment-effect plot.

In [ ]:
unt = augmented[augmented.treated.eq(0)].copy()
unt["resid0"] = unt.y - unt.y0_hat
unt.groupby("time")["resid0"].agg(["mean","std","count"]).round(3)

## Common failure

Imputation is not magic matrix completion. If untreated potential outcomes do not obey the posited parallel-trends/FE structure, the counterfactual can be wrong with very small residual standard errors. Identification still comes from assumptions, not prediction accuracy alone.

## Researcher failure checklist

- Fit the untreated model on untreated observations only.
- Verify support: every treated unit needs usable untreated history, and calendar times need untreated information.
- Pre-specify which treated observations enter the target weighted average.
- Test identifying restrictions separately from estimating treatment effects.
- Use the authors' production implementation for valid publication-grade covariance estimates.